# 🐼 Cookbook: encrypted dataframes

`pdpg.encrypt` takes a pandas DataFrame and returns a `CipherFrame`: named
columns over ciphertext. Column names travel inside the `.enc` file; values
never travel in the clear. Analyst habits — select by name, computed
columns, labeled aggregates — keep working.

In [ ]:
%%time
%pip install -q git+https://github.com/PDPG-lab/pypdpg

In [ ]:
import numpy as np
import pandas as pd
import pypdpg as pdpg

rng = np.random.default_rng(42)
df = pd.DataFrame({
    "income": rng.normal(58_000, 18_000, 500).clip(18_000, 150_000),
    "debt":   rng.normal(22_000, 12_000, 500).clip(0, 90_000),
    "age":    rng.uniform(21, 70, 500),
})

ctx = pdpg.Context.create()
enc = pdpg.encrypt(df, ctx)
enc

## Select by name, compute, stay encrypted

In [ ]:
monthly_income = enc["income"] / 12          # 1-D CipherArray
debt_k = enc["debt"] * 0.001
enc[["income", "debt"]]                      # sub-frame, names preserved

In [ ]:
# computed columns attach like they always did — still ciphertext
enc["debt_k"] = enc["debt"] * 0.001
enc

## Labeled aggregates

In [ ]:
means = enc.mean()     # encrypted, one scalar per column, labels intact
means

In [ ]:
# only the key holder sees numbers — as a real pandas Series
means.decrypt().round(2)

## Files keep their column names

In [ ]:
enc.save("portfolio.enc")
loaded = pdpg.load("portfolio.enc", ctx)
print(type(loaded).__name__, loaded.columns)

## What analysts cannot do (and why that's the product)

In [ ]:
try:
    enc["debt"] / enc["income"]     # debt-to-income ratio needs ciphertext division
except pdpg.EncryptedOperationError as e:
    print(f"⛔ {e}")

The ratio pattern under FHE: the *controller* decrypts the two aggregates
and divides plaintext — or the processor returns both encrypted columns and
lets the key holder do the division. Division is a read; reads need keys.

<sub>A [PDPG-lab](https://pdpglab.xyz) project. Current backend:
[TenSEAL](https://github.com/OpenMined/TenSEAL) (CKKS). More recipes in
[demo/cookbook](https://github.com/PDPG-lab/pypdpg/tree/main/demo/cookbook).</sub>